# Classic methods

## Setup

### Libraries import

In [24]:
import pandas as pd

from mlxtend.data import loadlocal_mnist

from sklearn import model_selection
from sklearn import metrics
from sklearn import tree
from sklearn import ensemble
from sklearn import neural_network

### Constants initialization

In [25]:
class config:
    # Data files paths
    DOWNLOADED_IMAGES_PATH = '../data/t10k-images.idx3-ubyte'
    DOWNLOADED_LABELS_PATH = '../data/t10k-labels.idx1-ubyte'

    # Format data config
    RANDOMIZE_DATA = True

    # General project settings
    FOLDS_CNT = 5

## Data setup

### Data load

In [26]:
# Load data from files
X, y = loadlocal_mnist(
    images_path=config.DOWNLOADED_IMAGES_PATH,
    labels_path=config.DOWNLOADED_LABELS_PATH
)

# Add names of columns
pixel_columns = [f"pixel{i}" for i in range(len(X[0]))]
df = pd.DataFrame(X, columns=pixel_columns)

# Add label column
df["label"] = y

# Show data
df

,pixel0,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,label
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,7
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,2
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,2
9996,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,3
9997,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,4
9998,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,5


## Model training

### Model dispatcher

In [57]:
class model_dispatcher:
    model_names = [
        "decision_tree_gini",
        "decision_tree_entropy",
        "random_forest",
        # "neural_network"
    ]

    models = {
        "decision_tree_gini": tree.DecisionTreeClassifier(
            criterion="gini"
        ),
        "decision_tree_entropy": tree.DecisionTreeClassifier(
            criterion="entropy"
        ),
        "random_forest": ensemble.RandomForestClassifier(),
        "neural_network": neural_network.MLPClassifier()
    }

### Training and testing models

In [58]:
def use_model(df_train, df_test, metrics_method):
    
    # Data configuration
    x_train = df_train.loc[:, df_train.columns != "label"].values
    y_train = df_train.loc[:, "label"].values

    x_test = df_test.loc[:, df_train.columns != "label"].values
    y_test = df_test.loc[:, "label"].values
    
    # Fit model
    model.fit(x_train, y_train)

    # Predict results
    predicted_data = model.predict(x_test)
    
    # Calculate and return accuracy score
    accuracy = metrics_method(
        y_true=y_test,
        y_pred=predicted_data
    )
    return accuracy

<img src="./images/image1.png" alt="image1" width="1300"/>
<!-- ![image1](./images/image1.png) -->
<!-- ![image1](https://towardsdatascience.com/wp-content/uploads/2023/12/1N45hocCMP0u4nXLe0WuSvw.png) -->

In [122]:
# Initialize dictionary with accuracy results
results_dict = {}

# Loop through all models
for model_name in model_dispatcher.model_names:
    model = model_dispatcher.models[model_name]
    
    results_dict[model_name] = []

    # Init accuracy sum
    accuracy_sum = 0
    
    # Test single model on different folds
    folds_model = model_selection.StratifiedKFold(
        n_splits=config.FOLDS_CNT,
        shuffle=config.RANDOMIZE_DATA
    )
    for fold, (train, test) in enumerate(folds_model.split(df, df["label"])):
        
        # Initialize train dataframe and test dataframe
        df_train = df.loc[train, :]
        df_test = df.loc[test, :]

        # Calculate, add to dictionary and print accuracy score
        accuracy = use_model(df_train, df_test, metrics.accuracy_score)
        results_dict[model_name].append(accuracy)
        
        print(f"{model_name}: {fold} - {accuracy:.3f}")

decision_tree_gini: 0 - 0.795
decision_tree_gini: 1 - 0.828
decision_tree_gini: 2 - 0.802
decision_tree_gini: 3 - 0.805
decision_tree_gini: 4 - 0.797
decision_tree_entropy: 0 - 0.816
decision_tree_entropy: 1 - 0.807
decision_tree_entropy: 2 - 0.814
decision_tree_entropy: 3 - 0.824
decision_tree_entropy: 4 - 0.813
random_forest: 0 - 0.950
random_forest: 1 - 0.948
random_forest: 2 - 0.952
random_forest: 3 - 0.959
random_forest: 4 - 0.957


In [63]:
results_dict

{'decision_tree_gini': [0.7995, 0.807, 0.807, 0.8195, 0.8035],
 'decision_tree_entropy': [0.824, 0.8075, 0.81, 0.822, 0.808],
 'random_forest': [0.955, 0.9515, 0.954, 0.953, 0.9515]}

In [119]:
accuracy_df = pd.DataFrame(results_dict)

display(accuracy_df)
display(accuracy_df.mean().to_frame().transpose())

,decision_tree_gini,decision_tree_entropy,random_forest
0,0.7995,0.8240,0.9550
1,0.8070,0.8075,0.9515
2,0.8070,0.8100,0.9540
3,0.8195,0.8220,0.9530
4,0.8035,0.8080,0.9515


,decision_tree_gini,decision_tree_entropy,random_forest
0,0.8073,0.8143,0.953


In [121]:
accuracy_df_tran = accuracy_df.transpose()

display(accuracy_df_tran)
display(accuracy_df_tran.mean(axis=1).to_frame())

,0,1,2,3,4
decision_tree_gini,0.7995,0.8070,0.807,0.8195,0.8035
decision_tree_entropy,0.8240,0.8075,0.810,0.8220,0.8080
random_forest,0.9550,0.9515,0.954,0.9530,0.9515


,0
decision_tree_gini,0.8073
decision_tree_entropy,0.8143
random_forest,0.9530
